# Medical Report Classification Using NLP

This notebook implements an NLP-based system to classify medical reports into:
- **Cardiology**
- **Neurology**
- **Oncology**
- **Orthopedics**

**Pipeline:** Text Preprocessing → TF-IDF → Machine Learning (Naive Bayes / SVM / Random Forest)

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import TextPreprocessor
from train_models import load_dataset, train_and_evaluate, predict_category

DATA_PATH = PROJECT_ROOT / "data" / "medical_reports.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load Dataset

In [ ]:
df = load_dataset(str(DATA_PATH))
print(f"Total reports: {len(df)}")
df.head()

In [ ]:
df["category"].value_counts().plot(kind="bar", color="steelblue", title="Reports per Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Text Preprocessing Example

In [ ]:
sample = "The patient is suffering from severe cardiac disease."
preprocessor = TextPreprocessor()
print("Original:", sample)
print("Processed:", preprocessor.clean_text(sample))

## 4. Train and Evaluate Models

In [ ]:
results, best_model = train_and_evaluate(df)

comparison = pd.DataFrame(
    {"Algorithm": list(results.keys()), "Accuracy (%)": [r.accuracy * 100 for r in results.values()]}
)
comparison

In [ ]:
best_name = max(results, key=lambda k: results[k].accuracy)
print(f"Best Model: {best_name}")
print(results[best_name].report)

## 5. Confusion Matrix (Best Model)

In [ ]:
best = results[best_name]
plt.figure(figsize=(8, 6))
sns.heatmap(best.confusion, annot=True, fmt="d", cmap="Blues", xticklabels=best.labels, yticklabels=best.labels)
plt.title(f"Confusion Matrix - {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 6. Predict New Medical Reports

In [ ]:
test_reports = [
    "ECG shows arrhythmia and elevated troponin levels.",
    "Patient has seizures and abnormal EEG findings.",
    "Biopsy confirms carcinoma requiring oncology referral.",
    "Fracture of the tibia visible on radiograph.",
]

for report in test_reports:
    print(f"Report: {report}")
    print(f"Predicted: {predict_category(best_model, report)}\n")